# Analisador BirdNET do Arbimon

Este notebook baixa gravações dos seus projetos no **Arbimon** para executar o **BirdNET v2.4** e detectar espécies de aves em cada gravação.

---

### O que este notebook faz (em ordem):
1. **Conecta ao seu Google Drive** para ler/salvar arquivos
2. **Instala os softwares necessários** automaticamente
3. **Faz login no Arbimon** para acessar suas gravações
4. **Baixa o áudio** dos pontos de gravação e janelas de tempo escolhidos
5. **Carrega o BirdNET** e cria um filtro opcional de espécies baseado na localização
6. **Analisa cada gravação** e salva os resultados como arquivos CSV no seu Drive

### Antes de começar:
- Você precisa de uma **conta Google** com Google Drive
- Você precisa de uma **conta no Arbimon** (em [arbimon.org](https://arbimon.org)) com acesso ao seu projeto
- Você precisa dos **IDs de stream** de cada ponto de gravação (veja o Passo 2 para saber como encontrá-los)
- A **latitude e longitude** dos seus pontos de gravação são recomendadas para o filtro de localização (opcional, mas recomendado)

### Como executar:
Execute cada célula **uma de cada vez**, de cima para baixo. Clique no botão ▶ à esquerda de cada célula ou pressione `Shift + Enter`.

> **Novo em notebooks?** Uma célula com `[ ]` à esquerda ainda não foi executada. Após executar, aparece `[1]`, `[2]`, etc. Se houver um erro (texto vermelho), leia a mensagem — ela geralmente indica exatamente o que corrigir.

---

Criado pela [biodiversica](https://biodiversica.xyz). Para problemas, dúvidas ou feedback, abra uma issue no [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) ou visite [biodiversica.xyz](https://biodiversica.xyz).

---
## Passo 1 — Conectar ao Google Drive e Instalar Software

Execute as duas células abaixo. A primeira pedirá que você **permita o acesso ao seu Google Drive** — clique no link e siga os passos.

In [ ]:
#@title 📂 Conectar ao Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado com sucesso.')

In [ ]:
#@title 📦 Instalar pacotes { display-mode: "form" }

!wget -q https://github.com/rfcx/rfcx-sdk-python/releases/download/0.3.1/rfcx-0.3.1-py3-none-any.whl \
    -O /tmp/rfcx-0.3.1-py3-none-any.whl

!pip install /tmp/rfcx-0.3.1-py3-none-any.whl -q
!pip install birdnet -q
!apt-get install -y -q libsndfile1

print('\nTodos os pacotes foram instalados com sucesso.')

---
## Passo 2 — Fazer login no Arbimon

A próxima célula fará login na plataforma Arbimon / RFCx.

**O que esperar:**
- Na primeira execução, aparecerá uma **URL e um código curto**. Abra essa URL no navegador e insira o código para autorizar o acesso. Esta é a mesma conta usada para entrar em [arbimon.org](https://arbimon.org).
- Suas credenciais serão **salvas no Google Drive** no caminho definido abaixo, e execuções futuras farão login automaticamente.

**Defina o caminho onde deseja salvar suas credenciais no Google Drive:**

In [ ]:
#@title 🔑 Fazer login { display-mode: "form" }
CREDENTIALS_PATH = '/content/drive/MyDrive/rfcx/.rfcx_credentials'  #@param {type:"string"}

import os
import rfcx

os.makedirs(os.path.dirname(CREDENTIALS_PATH), exist_ok=True)

client = rfcx.Client()

if os.path.exists(CREDENTIALS_PATH):
    print('Arquivo de credenciais encontrado — fazendo login automaticamente...')
    client.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
else:
    print('Nenhuma credencial salva encontrada.')
    print('Uma URL aparecerá abaixo. Abra-a no seu navegador e faça login com sua conta do Arbimon.')
    print('-' * 70)
    client.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    print('-' * 70)
    print(f'Credenciais salvas em: {CREDENTIALS_PATH}')
    print('Na próxima vez, o login será feito automaticamente.')

print('\nLogin no Arbimon realizado com sucesso.')

### Opcional: Encontrar os IDs de stream do Arbimon

Se você não souber o **stream ID** de cada ponto de gravação no seu projeto do Arbimon, execute a célula abaixo. Ela listará todos os projetos e pontos disponíveis na sua conta.

> Você precisará do **stream ID** (o código curto ao lado do nome de cada ponto) para preencher a configuração no Passo 3.

In [ ]:
#@title 🔍 Listar meus projetos e pontos no Arbimon { display-mode: "form" }

print('Buscando seus projetos e pontos de gravação no Arbimon...')
print('=' * 70)

projects = client.projects()
if not projects:
    print('Nenhum projeto encontrado. Verifique se você está logado na conta correta.')
else:
    for project in projects:
        print(f"\nPROJECT: {project['name']}  (id: {project['id']})")
        print('  Pontos de gravação (streams):')
        try:
            streams = client.streams(projects=project['id'])
            if streams:
                for stream in streams:
                    print(f"    stream_id: '{stream['id']}',  name: '{stream['name']}'")
            else:
                print('    (nenhum ponto encontrado neste projeto)')
        except Exception as e:
            print(f'    (não foi possível carregar os pontos: {e})')

---
## Passo 3 — Configuração

Preencha os formulários abaixo. **Não é necessário editar nenhum código** — basta digitar ou selecionar os valores em cada formulário e executar a célula.

Cada célula de formulário tem um botão ▶ pequeno à esquerda. Execute todos de cima para baixo:
1. **Configurações Gerais** — limiar de confiança, localização, pasta de resultados
2. **Pontos de Gravação** — um formulário por ponto (até 4); deixe em branco para ignorar

> **Dica:** Os formulários ocultam o código automaticamente. Para ver o código subjacente, clique no ícone `{ }` no canto superior direito de qualquer célula de formulário.

In [ ]:
#@title ⚙️ Configurações Gerais { display-mode: "form" }

import os

#@markdown **Pasta de resultados** — where detection results will be saved on your Google Drive.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/arbimon_birdnet_results"  #@param {type:"string"}

#@markdown ---
#@markdown **Limiar de detecção** — minimum confidence score to save a detection (0.0–1.0).
#@markdown Menor = mais detecções (pode incluir falsos positivos). Maior = menos, porém mais confiáveis.
MIN_CONFIDENCE = 0.5  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown ---
#@markdown **Top K detecções por segmento** — número máximo de detecções a manter por segmento de áudio.
#@markdown 1 = apenas a detecção com maior confiança por segmento. Aumente para manter mais.
TOP_K = 1  #@param {type:"integer"}

#@markdown ---
#@markdown **Localização do ponto de gravação** — usada pelo modelo geográfico do BirdNET para filtrar espécies improváveis
#@markdown nessa localização, reduzindo falsos positivos. Use graus decimais.
#@markdown Se seus pontos forem distantes entre si, use o ponto central aproximado da sua área de estudo.
LATITUDE  = -20.0  #@param {type:"number"}
LONGITUDE = -40.0  #@param {type:"number"}

#@markdown ---
#@markdown **Pontuação do filtro geográfico** — probabilidade mínima para uma espécie ser incluída na
#@markdown lista baseada em localização. Menor = mais inclusivo. Defina 0,0 para desativar o filtro
#@markdown e detectar todas as ~6.500 espécies do BirdNET.
GEO_MIN_SCORE = 0.03  #@param {type:"slider", min:0.0, max:1.0, step:0.01}

#@markdown ---
#@markdown **Sobreposição de segmentos (0,0–0,9)** — fração de sobreposição entre trechos consecutivos de 3 segundos do BirdNET.
#@markdown 0,0 = sem sobreposição (padrão). 0,5 = 50% de sobreposição (passo de 1,5 s, maior cobertura).
SEGMENT_OVERLAP = 0.0  #@param {type:"slider", min:0.0, max:0.9, step:0.1}

AUDIO_DIR = '/content/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f"Pasta de resultados  : {DRIVE_RESULTS_DIR}")
print(f"Confiança mínima   : {MIN_CONFIDENCE}")
print(f"Top K              : {TOP_K}")
print(f"Localização          : lat={LATITUDE}, lon={LONGITUDE}")
print(f"Filtro geográfico    : {GEO_MIN_SCORE}{'  (desativado — todas as espécies)' if GEO_MIN_SCORE == 0.0 else ''}")
print(f"Sobreposição         : {SEGMENT_OVERLAP}")

In [ ]:
#@title ⚙️ Pré-processamento de Áudio { display-mode: "form" }
#@markdown Optional preprocessing applied to each recording before inference.
#@markdown
#@markdown ---
#@markdown **Filtro de frequência** — remove frequências fora da faixa de interesse.
FILTER_TYPE = "none"  #@param ["none", "lowpass", "highpass", "bandpass"]

#@markdown **Frequência de corte inferior (Hz)** — usada nos filtros passa-alta e passa-banda.
FILTER_LOW_HZ = 0  #@param {type:"integer"}

#@markdown **Frequência de corte superior (Hz)** — usada nos filtros passa-baixa e passa-banda.
FILTER_HIGH_HZ = 15000  #@param {type:"integer"}

#@markdown ---
#@markdown **Velocidade de reprodução** — 1.0 = normal. Abaixo de 1.0 desacelera e alonga o áudio;
#@markdown acima de 1.0 acelera e encurta. Útil para deslocar o conteúdo de frequência
#@markdown para a faixa esperada pelo modelo (ex.: 0.5x divide todas as frequências pela metade).
AUDIO_SPEED = 1.0  #@param {type:"slider", min:0.25, max:4.0, step:0.05}

_preprocess_lines = []
if FILTER_TYPE != 'none':
    if FILTER_TYPE == 'lowpass':
        _preprocess_lines.append(f'Filter   : lowpass <= {FILTER_HIGH_HZ} Hz')
    elif FILTER_TYPE == 'highpass':
        _preprocess_lines.append(f'Filter   : highpass >= {FILTER_LOW_HZ} Hz')
    elif FILTER_TYPE == 'bandpass':
        _preprocess_lines.append(f'Filter   : bandpass {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz')
if AUDIO_SPEED != 1.0:
    _preprocess_lines.append(f'Speed    : {AUDIO_SPEED}x')
if _preprocess_lines:
    print('Pré-processamento ativado:')
    for _l in _preprocess_lines:
        print(f'  {_l}')
else:
    print('Pré-processamento : nenhum')

#@markdown ---
#@markdown **Filtro de horário** — analisa apenas gravações dentro de uma janela de tempo diária específica.
#@markdown Deixe desativado para analisar todas as gravações independentemente do horário.
TIME_FILTER_ENABLED = False  #@param {type:"boolean"}
#@markdown **Início da janela (HH:MM)**
TIME_FILTER_START = "17:00"  #@param {type:"string"}
#@markdown **Fim da janela (HH:MM)** — pode cruzar a meia-noite, ex.: 22:00 a 06:00.
TIME_FILTER_END = "18:00"  #@param {type:"string"}

if TIME_FILTER_ENABLED:
    print(f'Filtro de horário : {TIME_FILTER_START} – {TIME_FILTER_END}')

In [ ]:
#@title 📍 Ponto de Gravação 1 { display-mode: "form" }

#@markdown **Stream ID** — o código curto deste ponto de gravação (da célula opcional no Passo 2).
stream_id_1 = ""  #@param {type:"string"}
#@markdown **Data e hora de início** (horário local)
min_date_1 = "2025-12-01"  #@param {type:"date"}
min_time_1 = "00:00"  #@param {type:"string"}
#@markdown **Data e hora de fim** (horário local)
max_date_1 = "2025-12-03"  #@param {type:"date"}
max_time_1 = "23:59"  #@param {type:"string"}
if stream_id_1.strip():
    print(f"Ponto 1: '{stream_id_1}' — {min_date_1} {min_time_1} → {max_date_1} {max_time_1}")
else:
    print("Ponto 1: nenhum Stream ID informado.")

### 📍 Pontos de Gravação Adicionais (opcional)
> Precisa analisar mais de um ponto? Expanda esta seção para configurar até 3 pontos adicionais.
> Deixe o **Stream ID** em branco em qualquer ponto que não precisar. Execute novamente 'Confirmar lista de pontos' após adicionar pontos opcionais.

In [ ]:
#@title 📍 Ponto de Gravação 2 (opcional) { display-mode: "form" }

#@markdown Deixe o **Stream ID** em branco para ignorar este ponto.
stream_id_2 = ""  #@param {type:"string"}
min_date_2 = "2025-01-01"  #@param {type:"date"}
min_time_2 = "18:00"  #@param {type:"string"}
max_date_2 = "2025-01-02"  #@param {type:"date"}
max_time_2 = "06:00"  #@param {type:"string"}
if stream_id_2.strip():
    print(f"Ponto 2: '{stream_id_2}' — {min_date_2} {min_time_2} → {max_date_2} {max_time_2}")
else:
    print("Ponto 2: vazio — será ignorado.")

In [ ]:
#@title 📍 Ponto de Gravação 3 (opcional) { display-mode: "form" }

#@markdown Deixe o **Stream ID** em branco para ignorar este ponto.
stream_id_3 = ""  #@param {type:"string"}
min_date_3 = "2025-01-01"  #@param {type:"date"}
min_time_3 = "18:00"  #@param {type:"string"}
max_date_3 = "2025-01-02"  #@param {type:"date"}
max_time_3 = "06:00"  #@param {type:"string"}
if stream_id_3.strip():
    print(f"Ponto 3: '{stream_id_3}' — {min_date_3} {min_time_3} → {max_date_3} {max_time_3}")
else:
    print("Ponto 3: vazio — será ignorado.")

In [ ]:
#@title 📍 Ponto de Gravação 4 (opcional) { display-mode: "form" }

#@markdown Deixe o **Stream ID** em branco para ignorar este ponto.
stream_id_4 = ""  #@param {type:"string"}
min_date_4 = "2025-01-01"  #@param {type:"date"}
min_time_4 = "18:00"  #@param {type:"string"}
max_date_4 = "2025-01-02"  #@param {type:"date"}
max_time_4 = "06:00"  #@param {type:"string"}
if stream_id_4.strip():
    print(f"Ponto 4: '{stream_id_4}' — {min_date_4} {min_time_4} → {max_date_4} {max_time_4}")
else:
    print("Ponto 4: vazio — será ignorado.")

In [ ]:
#@title ✅ Confirmar lista de pontos { display-mode: "form" }
#@markdown Execute esta célula após preencher os formulários de pontos acima.
#@markdown Ela mostrará um resumo de todos os pontos que serão analisados.

import datetime

_empty = ("", "2025-01-01", "00:00", "2025-01-01", "00:00")
try: stream_id_2
except NameError: stream_id_2, min_date_2, min_time_2, max_date_2, max_time_2 = _empty
try: stream_id_3
except NameError: stream_id_3, min_date_3, min_time_3, max_date_3, max_time_3 = _empty
try: stream_id_4
except NameError: stream_id_4, min_date_4, min_time_4, max_date_4, max_time_4 = _empty

def _build_job(stream_id, min_date_str, min_time_str, max_date_str, max_time_str):
    if not stream_id.strip():
        return None
    def _parse(date_str, time_str):
        d = datetime.date.fromisoformat(date_str.strip())
        parts = time_str.strip().split(':')
        h, m = int(parts[0]), int(parts[1])
        s = int(parts[2]) if len(parts) > 2 else 0
        return datetime.datetime(d.year, d.month, d.day, h, m, s)
    return {
        'stream_id':        stream_id.strip(),
        'stream_name':      stream_id.strip(),  # replaced from API below
        'min_date':         _parse(min_date_str, min_time_str),
        'max_date':         _parse(max_date_str, max_time_str),
        'lat':              '',
        'lon':              '',
        'timezone_str':     None,
        'utc_offset_hours': 0,
    }

def _tz_to_utc_offset(timezone_value, reference_dt=None):
    if timezone_value is None or timezone_value == '':
        return None
    try:
        return float(timezone_value)
    except (ValueError, TypeError):
        pass
    try:
        from zoneinfo import ZoneInfo
        import datetime as _dt
        tz = ZoneInfo(str(timezone_value))
        ref = (reference_dt or _dt.datetime.utcnow()).replace(tzinfo=_dt.timezone.utc)
        offset = ref.astimezone(tz).utcoffset()
        return offset.total_seconds() / 3600
    except Exception:
        return None

JOBS = []
for args in [
    (stream_id_1, min_date_1, min_time_1, max_date_1, max_time_1),
    (stream_id_2, min_date_2, min_time_2, max_date_2, max_time_2),
    (stream_id_3, min_date_3, min_time_3, max_date_3, max_time_3),
    (stream_id_4, min_date_4, min_time_4, max_date_4, max_time_4),
]:
    job = _build_job(*args)
    if job:
        JOBS.append(job)

_all_streams = client.streams() or []
_stream_map = {s['id']: s for s in _all_streams if isinstance(s, dict) and 'id' in s}
for job in JOBS:
    s = _stream_map.get(job['stream_id'], {})
    job['stream_name'] = s.get('name', job['stream_id']).replace(' ', '_')
    job['lat'] = s.get('latitude') or s.get('lat', '') or ''
    job['lon'] = s.get('longitude') or s.get('lon', '') or ''
    tz_val = s.get('timezone', None)
    job['timezone_str'] = str(tz_val) if tz_val else None
    mid_dt = job['min_date'] + (job['max_date'] - job['min_date']) / 2
    utc_off = _tz_to_utc_offset(tz_val, reference_dt=mid_dt)
    if utc_off is not None:
        job['utc_offset_hours'] = utc_off
    else:
        print(f"  AVISO: fuso horário não encontrado para {job['stream_name']} ({job['stream_id']}) — usando UTC+0")
    if not job['lat'] and not job['lon']:
        print(f"  AVISO: coordenadas não encontradas para {job['stream_name']} ({job['stream_id']})")

if not JOBS:
    print("AVISO: Nenhum ponto de gravação configurado. Preencha pelo menos um formulário acima.")
else:
    print(f"{len(JOBS)} ponto(s) pronto(s):")
    for j in JOBS:
        utc_off = j['utc_offset_hours']
        print(f"  {j['stream_name']} ({j['stream_id']})")
        print(f"    {j['min_date'].strftime('%Y-%m-%d %H:%M')} → {j['max_date'].strftime('%Y-%m-%d %H:%M')}  (local UTC{utc_off:+.0f})  lat={j['lat']} lon={j['lon']}")
    print("\nConfiguração concluída. Continue para o Passo 4.")

---
## Passo 4 — Baixar áudio do Arbimon

Esta célula conecta ao Arbimon e baixa todas as gravações nas janelas de tempo definidas acima.

> Os arquivos são baixados para uma **pasta temporária dentro do Colab** (`/content/audio`). Eles não são salvos no seu Google Drive. Se a sessão reiniciar, será necessário executar este passo novamente.

In [ ]:
#@title 🗑️ Limpar áudio baixado (opcional) { display-mode: "form" }
#@markdown Execute esta célula para apagar todos os arquivos da pasta temporária de áudio e liberar espaço no Colab.
#@markdown Isso **não** afeta seu Google Drive nem os arquivos de resultados.

import glob as _g

files = _g.glob(f'{AUDIO_DIR}/**/*', recursive=True)
files = [f for f in files if os.path.isfile(f)]

if not files:
    print('A pasta de áudio já está vazia.')
else:
    for f in files:
        os.remove(f)
    for d in sorted(_g.glob(f'{AUDIO_DIR}/*/'), reverse=True):
        try:
            os.rmdir(d)
        except OSError:
            pass
    print(f'Excluídos {len(files)} arquivo(s) de {AUDIO_DIR}.')

In [ ]:
#@title ⬇️ Baixar áudio do Arbimon { display-mode: "form" }
#@markdown Baixa todas as gravações para os pontos e janelas de tempo configurados no Passo 3.
#@markdown Os arquivos são armazenados temporariamente no Colab — não são salvos no Drive.

import os, glob, datetime

os.makedirs(AUDIO_DIR, exist_ok=True)

for job in JOBS:
    utc_off     = job['utc_offset_hours']
    job_utc_min = job['min_date'] - datetime.timedelta(hours=utc_off)
    job_utc_max = job['max_date'] - datetime.timedelta(hours=utc_off)

    print(f"\n{'─' * 65}")
    print(f"Site   : {job['stream_name']}  (id: {job['stream_id']})")
    print(f"Window : {job['min_date'].strftime('%Y-%m-%d %H:%M')} → {job['max_date'].strftime('%Y-%m-%d %H:%M')}  (local UTC{utc_off:+.0f})")

    if TIME_FILTER_ENABLED:
        ps = TIME_FILTER_START.split(':')
        pe = TIME_FILTER_END.split(':')
        t_h_s, t_m_s = int(ps[0]), int(ps[1])
        t_h_e, t_m_e = int(pe[0]), int(pe[1])
        crosses_midnight = (t_h_e, t_m_e) < (t_h_s, t_m_s)
        current_day = job['min_date'].date()
        end_day     = job['max_date'].date()
        print(f"Time   : {TIME_FILTER_START}–{TIME_FILTER_END} (local) → downloading per-day windows")
        while current_day <= end_day:
            local_win_s = datetime.datetime(current_day.year, current_day.month, current_day.day, t_h_s, t_m_s)
            if not crosses_midnight:
                local_win_e = datetime.datetime(current_day.year, current_day.month, current_day.day, t_h_e, t_m_e)
            else:
                _nd = current_day + datetime.timedelta(days=1)
                local_win_e = datetime.datetime(_nd.year, _nd.month, _nd.day, t_h_e, t_m_e)
            utc_win_s = max(local_win_s - datetime.timedelta(hours=utc_off), job_utc_min)
            utc_win_e = min(local_win_e - datetime.timedelta(hours=utc_off), job_utc_max)
            if utc_win_s < utc_win_e:
                print(f"  {current_day}  {TIME_FILTER_START}–{TIME_FILTER_END} local → downloading...")
                try:
                    client.download_segments(
                        dest_path=AUDIO_DIR,
                        stream=job['stream_id'],
                        min_date=utc_win_s,
                        max_date=utc_win_e,
                        parallel=True,
                    )
                except Exception as e:
                    print(f"  ERROR: {e}")
            current_day += datetime.timedelta(days=1)
    else:
        print(f"Downloading...")
        try:
            client.download_segments(
                dest_path=AUDIO_DIR,
                stream=job['stream_id'],
                min_date=job_utc_min,
                max_date=job_utc_max,
                parallel=True,
            )
        except Exception as e:
            print(f"  ERROR: {e}")
            print("  Check that the stream_id is correct and you have access to this site.")

all_audio = sorted(
    glob.glob(f'{AUDIO_DIR}/**/*.wav',  recursive=True) +
    glob.glob(f'{AUDIO_DIR}/**/*.flac', recursive=True) +
    glob.glob(f'{AUDIO_DIR}/**/*.mp3',  recursive=True)
)

print(f"\n{'─' * 65}")
print(f"Total audio files downloaded: {len(all_audio)}")
for f in all_audio[:10]:
    print(f"  {f.replace(AUDIO_DIR + '/', '')}")
if len(all_audio) > 10:
    print(f"  ... and {len(all_audio) - 10} more")

---
## Passo 5 — Carregar o BirdNET

Este passo carrega dois componentes do BirdNET:

1. **Modelo geográfico** — pontua a probabilidade de ocorrência de cada uma das ~6.500 espécies do BirdNET na sua localização e época do ano. Espécies abaixo de `GEO_MIN_SCORE` são excluídas, reduzindo falsos positivos. Defina `GEO_MIN_SCORE = 0.0` nas Configurações Gerais para desativar esse filtro e detectar todas as espécies.
2. **Modelo acústico** — a rede neural principal que analisa segmentos de áudio de 3 segundos e pontua cada espécie.

> Esta célula pode demorar um minuto na primeira execução enquanto o BirdNET baixa seus pesos.

In [ ]:
#@title 🐦 Carregar modelo { display-mode: "form" }

import birdnet
import tempfile

MODEL_NAME = 'BirdNET_v2.4'

species_list_path = None

if GEO_MIN_SCORE > 0.0:
    week = min(JOBS[0]['min_date'].isocalendar()[1], 48)
    print(f'Executando modelo geográfico: lat={LATITUDE}, lon={LONGITUDE}, semana={week} ...')
    geo_model = birdnet.load('geo', '2.4', 'tf')
    geo_preds = geo_model.predict(LATITUDE, LONGITUDE, week=week)

    geo_df     = geo_preds.to_dataframe(sort_by='confidences')
    species_df = geo_df[geo_df['confidence'] >= GEO_MIN_SCORE]
    print(f'Espécies no filtro de localização: {len(species_df)}')

    tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
    tmp.write('\n'.join(species_df['species_name'].tolist()))
    tmp.close()
    species_list_path = tmp.name
    print(f'Lista de espécies gravada em: {species_list_path}')
else:
    print('Filtro de localização desativado — todas as ~6.500 espécies do BirdNET serão consideradas.')

print()
print('Carregando modelo acústico ...')
acoustic_model = birdnet.load('acoustic', '2.4', 'tf')
print(f'Modelo acústico do BirdNET pronto.')
print(f'Nome do modelo : {MODEL_NAME}')

---
## Passo 6 — Verificar análises anteriores

Esta célula verifica sua pasta de resultados para ver quais arquivos de áudio **já foram analisados**. Esses arquivos serão **ignorados** no próximo passo, para que você possa executar a análise novamente sem repetir o trabalho já feito.

In [ ]:
#@title 🔎 Verificar{ display-mode: "form" }

import os
import re
import glob as _glob
from datetime import datetime, timedelta

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

def parse_file_datetime(filename, utc_offset_hours=0, timezone_str=None):
    name = os.path.basename(filename)
    utc_dt = None
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        utc_dt = datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                          int(m.group(4)), int(m.group(5)), int(m.group(6)))
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
        if m:
            d, t = m.group(1), m.group(2)
            utc_dt = datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                              int(t[:2]), int(t[2:4]), int(t[4:6]))
    if utc_dt is None:
        return None
    if timezone_str:
        try:
            from zoneinfo import ZoneInfo
            local_dt = utc_dt.replace(tzinfo=ZoneInfo('UTC')).astimezone(ZoneInfo(timezone_str))
            return local_dt.replace(tzinfo=None)
        except Exception:
            pass
    return utc_dt + timedelta(hours=utc_offset_hours)

_job_lookup = {}
for _j in JOBS:
    _job_lookup[_j['stream_id']]   = _j
    _job_lookup[_j['stream_name']] = _j

def _resolve_job(audio_path):
    job = _job_lookup.get(os.path.basename(os.path.dirname(audio_path)))
    if job:
        return job
    for _j in JOBS:
        if _j['stream_id'] and _j['stream_id'] in audio_path:
            return _j
    return JOBS[0] if len(JOBS) == 1 else {}

def get_result_path(audio_path, results_dir, model_name):
    filename  = os.path.basename(audio_path)
    job       = _resolve_job(audio_path)
    site_name = job.get('stream_name', os.path.basename(os.path.dirname(audio_path)))
    utc_off   = job.get('utc_offset_hours', 0)
    tz_str    = job.get('timezone_str', None)
    file_dt   = parse_file_datetime(filename, utc_offset_hours=utc_off, timezone_str=tz_str)
    if file_dt is not None:
        dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
        result_fn = f"{site_name}_{dt_str}.{model_name}.results.csv"
    else:
        file_base = os.path.splitext(filename)[0]
        result_fn = f"{site_name}_{file_base}.{model_name}.results.csv"
    return os.path.join(results_dir, site_name, result_fn)

all_audio = sorted(
    _glob.glob(f'{AUDIO_DIR}/**/*.wav',  recursive=True) +
    _glob.glob(f'{AUDIO_DIR}/**/*.flac', recursive=True) +
    _glob.glob(f'{AUDIO_DIR}/**/*.mp3',  recursive=True)
)

def _time_filter_ok(audio_path):
    if not TIME_FILTER_ENABLED:
        return True
    _job  = _resolve_job(audio_path)
    _fdt  = parse_file_datetime(
        os.path.basename(audio_path),
        utc_offset_hours=_job.get('utc_offset_hours', 0),
        timezone_str=_job.get('timezone_str', None)
    )
    if _fdt is None:
        return True
    from datetime import time as _dtime
    _t  = _fdt.time()
    _ps = TIME_FILTER_START.split(':')
    _pe = TIME_FILTER_END.split(':')
    _ts = _dtime(int(_ps[0]), int(_ps[1]))
    _te = _dtime(int(_pe[0]), int(_pe[1]))
    return (_ts <= _t <= _te) if _ts <= _te else (_t >= _ts or _t <= _te)

to_process   = [f for f in all_audio
                if not os.path.exists(get_result_path(f, DRIVE_RESULTS_DIR, MODEL_NAME))
                and _time_filter_ok(f)]
already_done = [f for f in all_audio if f not in to_process]

print(f'Pasta de resultados : {DRIVE_RESULTS_DIR}')
print(f'Nome do modelo      : {MODEL_NAME}')
print()
print(f'Total de arquivos   : {len(all_audio)}')
print(f'Já analisados       : {len(already_done)}')
print(f'Restando analisar   : {len(to_process)}')

if already_done:
    print()
    print('Já concluídos (serão ignorados):')
    for f in already_done:
        rp = get_result_path(f, DRIVE_RESULTS_DIR, MODEL_NAME)
        print(f"  {os.path.basename(f)}  →  {os.path.basename(rp)}")

if not to_process:
    print()
    print('Todos os arquivos já foram analisados. Nada a fazer!')
    print('Para re-analisar, exclua primeiro os arquivos de resultados correspondentes do seu Drive.')
else:
    print()
    print(f'Pronto para analisar {len(to_process)} arquivo(s). Prossiga para o Passo 7.')

---
## Passo 7 — Executar a análise e salvar resultados

Este é o passo principal de análise. Para cada arquivo de áudio, o BirdNET divide a gravação em segmentos de 3 segundos, pontua cada espécie e salva as detecções acima do seu limiar de confiança.

**Arquivos de resultado** são salvos como:
```
DRIVE_RESULTS_DIR / NOME_DO_PONTO / arquivo_audio.BirdNET_v2.4.results.csv
```

Cada arquivo de resultados tem as colunas: `stream_id`, `site`, `date`, `time`, `utc_offset`, `start_time`, `end_time`, `scientific_name`, `common_name`, `confidence`

- `start_time` / `end_time` — deslocamento em segundos desde o início do arquivo de áudio para cada segmento detectado.
- `date` / `time` — timestamp local absoluto da detecção, derivado do nome do arquivo de gravação.
- `utc_offset` — o deslocamento UTC (em horas) aplicado para converter os timestamps das gravações para o horário local.

> Dependendo do número de arquivos e da duração das gravações, este passo pode demorar. O progresso é mostrado abaixo.

In [ ]:
#@title 🚀 Executar { display-mode: "form" }

import csv
import re
import time
import os
from datetime import datetime, timedelta

def parse_file_datetime(filename, utc_offset_hours=0, timezone_str=None):
    name = os.path.basename(filename)
    utc_dt = None
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        utc_dt = datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                          int(m.group(4)), int(m.group(5)), int(m.group(6)))
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
        if m:
            d, t = m.group(1), m.group(2)
            utc_dt = datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                              int(t[:2]), int(t[2:4]), int(t[4:6]))
    if utc_dt is None:
        return None
    if timezone_str:
        try:
            from zoneinfo import ZoneInfo
            local_dt = utc_dt.replace(tzinfo=ZoneInfo('UTC')).astimezone(ZoneInfo(timezone_str))
            return local_dt.replace(tzinfo=None)
        except Exception:
            pass
    return utc_dt + timedelta(hours=utc_offset_hours)

def _time_to_seconds(t):
    if hasattr(t, 'total_seconds'):
        return t.total_seconds()
    s = str(t)
    if ':' in s:
        parts = s.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    return float(s)


def _in_time_window(file_dt, start_str, end_str):
    from datetime import time as _dtime
    if file_dt is None:
        return True
    t = file_dt.time()
    ps, pe = start_str.split(':'), end_str.split(':')
    t_start = _dtime(int(ps[0]), int(ps[1]))
    t_end   = _dtime(int(pe[0]), int(pe[1]))
    if t_start <= t_end:
        return t_start <= t <= t_end
    return t >= t_start or t <= t_end

_job_lookup = {}
for _j in JOBS:
    _job_lookup[_j['stream_id']]   = _j
    _job_lookup[_j['stream_name']] = _j

def _resolve_job(audio_path):
    job = _job_lookup.get(os.path.basename(os.path.dirname(audio_path)))
    if job:
        return job
    for _j in JOBS:
        if _j['stream_id'] and _j['stream_id'] in audio_path:
            return _j
    return JOBS[0] if len(JOBS) == 1 else {}

total_detections = 0
total_start      = time.time()

if not to_process:
    print('Nenhum arquivo para processar. Todas as análises já estão concluídas.')
else:
    print(f'Iniciando análise BirdNET de {len(to_process)} arquivo(s)')
    print(f'Limiar de confiança  : {MIN_CONFIDENCE}')
    print(f'Pontuação geográfica : {GEO_MIN_SCORE}{"  (desativado)" if GEO_MIN_SCORE == 0.0 else ""}')
    if FILTER_TYPE != 'none' or AUDIO_SPEED != 1.0:
        _pp = []
        if FILTER_TYPE != 'none': _pp.append(f'filter={FILTER_TYPE}')
        if AUDIO_SPEED != 1.0:   _pp.append(f'speed={AUDIO_SPEED}x')
        print(f'Preprocessing        : {", ".join(_pp)}')
    print('=' * 65)

    for file_idx, audio_path in enumerate(to_process, 1):
        filename        = os.path.basename(audio_path)
        _job            = _resolve_job(audio_path)
        site_name       = _job.get('stream_name', os.path.basename(os.path.dirname(audio_path)))
        stream_id       = _job.get('stream_id', '')
        site_lat        = _job.get('lat', '')
        site_lon        = _job.get('lon', '')
        site_utc_offset = _job.get('utc_offset_hours', 0)
        tz_str          = _job.get('timezone_str', None)
        utc_offset_str  = f'UTC{site_utc_offset:+.0f}'
        file_dt         = parse_file_datetime(filename, utc_offset_hours=site_utc_offset, timezone_str=tz_str)
        if TIME_FILTER_ENABLED and not _in_time_window(file_dt, TIME_FILTER_START, TIME_FILTER_END):
            print(f'[{file_idx}/{len(to_process)}] {filename}  — skipped (outside {TIME_FILTER_START}–{TIME_FILTER_END})')
            continue

        if file_dt is not None:
            dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
            result_fn = f"{site_name}_{dt_str}.{MODEL_NAME}.results.csv"
        else:
            file_base = os.path.splitext(filename)[0]
            result_fn = f"{site_name}_{file_base}.{MODEL_NAME}.results.csv"
            print(f"  AVISO: não foi possível interpretar data/hora de '{filename}' — usando nome do arquivo.")

        print(f'[{file_idx}/{len(to_process)}] {filename}  (site: {site_name}, stream: {stream_id})')

        file_start = time.time()
        rows = []

        try:
            kwargs = {}
            if species_list_path:
                kwargs['custom_species_list'] = species_list_path
            if SEGMENT_OVERLAP > 0.0:
                kwargs['overlap_duration_s'] = round(SEGMENT_OVERLAP * 3.0, 1)
            if FILTER_TYPE == 'lowpass':
                kwargs['bandpass_fmax'] = FILTER_HIGH_HZ
            elif FILTER_TYPE == 'highpass':
                kwargs['bandpass_fmin'] = FILTER_LOW_HZ
            elif FILTER_TYPE == 'bandpass':
                kwargs['bandpass_fmin'] = FILTER_LOW_HZ
                kwargs['bandpass_fmax'] = FILTER_HIGH_HZ
            if AUDIO_SPEED != 1.0:
                kwargs['speed'] = AUDIO_SPEED
            result = acoustic_model.predict(audio_path, **kwargs)
            preds  = result.to_dataframe()
        except Exception as e:
            print(f'  ERROR during analysis: {e} — skipping.')
            continue

        if not preds.empty:
            preds = preds[preds['confidence'] >= MIN_CONFIDENCE].copy()
            if TOP_K > 0 and not preds.empty:
                preds = (preds.groupby(['start_time', 'end_time'], group_keys=False)
                              .apply(lambda g: g.nlargest(TOP_K, 'confidence')))
            for _, row in preds.iterrows():
                start_s = _time_to_seconds(row['start_time'])
                end_s   = _time_to_seconds(row['end_time'])

                sp = str(row['species_name']).split('_', 1)
                scientific = sp[0].strip()
                common     = sp[1].strip() if len(sp) > 1 else ''

                if file_dt is not None:
                    date_str = file_dt.strftime('%Y-%m-%d')
                    time_str = file_dt.strftime('%H:%M:%S')
                else:
                    date_str = ''
                    time_str = ''

                rows.append([
                    stream_id, site_name, site_lat, site_lon, date_str, time_str,
                    utc_offset_str,
                    f'{start_s:.1f}', f'{end_s:.1f}',
                    scientific, common, f'{float(row["confidence"]):.4f}'
                ])

        elapsed = time.time() - file_start

        site_dir    = os.path.join(DRIVE_RESULTS_DIR, site_name)
        os.makedirs(site_dir, exist_ok=True)
        result_path = os.path.join(site_dir, result_fn)

        with open(result_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['stream_id', 'site', 'lat', 'lon', 'date', 'time', 'utc_offset',
                             'start_time', 'end_time', 'scientific_name', 'common_name', 'confidence'])
            writer.writerows(rows)

        total_detections += len(rows)
        print(f'  -> {len(rows)} detection(s)  |  {elapsed:.1f}s  →  {result_fn}')

    total_elapsed = time.time() - total_start
    print()
    print('=' * 65)
    print(f'Analysis complete!')
    print(f'  Arquivos analisados  : {len(to_process)}')
    print(f'  Total de detecções   : {total_detections}')
    print(f'  Tempo total          : {total_elapsed:.1f}s')
    print(f'  Resultados salvos em : {DRIVE_RESULTS_DIR}')

---
## Passo 8 — Visualizar resultados (opcional)

Execute a célula abaixo para carregar todos os resultados salvos nesta sessão e exibir um resumo das espécies detectadas ordenado pelo número de detecções.

In [ ]:
#@title 📊 Resumo de detecções { display-mode: "form" }

import glob as _glob
import pandas as pd

result_files = _glob.glob(f'{DRIVE_RESULTS_DIR}/**/*.results.csv', recursive=True)

if not result_files:
    print(f'Nenhum arquivo de resultado encontrado em {DRIVE_RESULTS_DIR}.')
else:
    frames = [pd.read_csv(f) for f in result_files]
    df_all = pd.concat(frames, ignore_index=True)

    print(f'Arquivos de resultado carregados : {len(result_files)}')
    print(f'Total de detecções               : {len(df_all)}')
    print()

    if df_all.empty:
        print('Nenhuma detecção acima do limiar de confiança.')
    else:
        summary = (
            df_all.groupby(['scientific_name', 'common_name'])
                  .agg(detections=('confidence', 'count'),
                       mean_conf  =('confidence', 'mean'))
                  .sort_values('detections', ascending=False)
                  .reset_index()
        )
        summary['mean_conf'] = summary['mean_conf'].round(3)
        print(f'Espécies detectadas: {len(summary)}')
        display(summary)

---
## Mesclar Resultados em um Único CSV

O passo "Executar" acima salva um arquivo de resultado por gravação (para que uma execução interrompida
possa retomar). Este passo combina todos esses arquivos em um **único CSV** no seu Google Drive:
```
PASTA_RESULTADOS / MODEL_NAME.merged.results.csv
```
Colunas: `stream_id`, `site`, `lat`, `lon`, `date`, `time`, `utc_offset`, `start_time`, `end_time`, `scientific_name`, `common_name`, `confidence`

Carregue este arquivo no notebook **Análise de Resultados de Classificação** apontando a configuração
*Pasta de resultados* para este arquivo `.merged.results.csv`. (defina a coluna de rótulo como `common_name` ou `scientific_name`).

> Os arquivos individuais por gravação **não** são apagados — você pode reexecutar este passo quando quiser.

In [ ]:
#@title 🧩 Mesclar todos os resultados em um único CSV { display-mode: "form" }

#@markdown Combina todos os arquivos de resultado por gravação da pasta de resultados em um único CSV,
#@markdown pronto para carregar no notebook **Análise de Resultados de Classificação**.
#@markdown Os arquivos individuais por gravação são mantidos para que uma execução interrompida possa retomar.

import os
import csv
import glob

MERGED_CSV_PATH = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}.merged.results.csv')

# Encontra todos os arquivos de resultado por gravação (busca recursiva, um por subpasta de ponto).
_result_files = sorted(glob.glob(
    os.path.join(DRIVE_RESULTS_DIR, '**', f'*.{MODEL_NAME}.results.csv'), recursive=True))

if not _result_files:
    print(f'Nenhum arquivo de resultado encontrado em: {DRIVE_RESULTS_DIR}')
    print('Execute o passo de análise primeiro para gerar os resultados por gravação.')
else:
    # Processa arquivo por arquivo para nunca manter todos os resultados na memória de uma vez.
    _header  = None
    _n_files = 0
    _n_rows  = 0
    with open(MERGED_CSV_PATH, 'w', newline='', encoding='utf-8') as _out:
        _writer = csv.writer(_out)
        for _fp in _result_files:
            if os.path.abspath(_fp) == os.path.abspath(MERGED_CSV_PATH):
                continue  # nunca mesclar o arquivo de saída nele mesmo
            try:
                with open(_fp, 'r', newline='', encoding='utf-8') as _in:
                    _rows = list(csv.reader(_in))
            except Exception as _e:
                print(f'  AVISO: não foi possível ler {os.path.basename(_fp)}: {_e}')
                continue
            if not _rows:
                continue
            _file_header, _data = _rows[0], _rows[1:]
            if _header is None:
                _header = _file_header
                _writer.writerow(_header)
            _writer.writerows(_data)
            _n_files += 1
            _n_rows  += len(_data)

    print('Mesclagem concluída!')
    print(f'  Arquivos mesclados   : {_n_files}')
    print(f'  Total de detecções   : {_n_rows}')
    print(f'  CSV salvo em         : {MERGED_CSV_PATH}')